# 14 · Recuperación densa: primera ejecución con BGE-M3

| Archivo | Ubicación dentro del proyecto |
|---|---|
| Esta libreta | `07_RAG/Pruebas_RAG/14_recuperacion_densa.ipynb` |
| Catálogo | `06_grafos/catalogo_articulos.csv` |
| Consultas fijadas | `07_RAG/consultas_evaluacion_v1.csv` |
| Vectores existentes de artículos | `07_RAG/bge_m3_vectores_v1.npz` |
| Índice existente | `07_RAG/bge_m3_indice_v1.faiss` |
| Referencia léxica ya ejecutada | `07_RAG/Pruebas_RAG/bm25_resultados_v1.json` |
| Única salida nueva | `07_RAG/Pruebas_RAG/densa_resultados_v1.json` |

## 1. Leer las entradas fijadas

La libreta busca la raíz del proyecto desde su carpeta de ejecución. Comprueba
que catálogo, consultas y BM25 pertenecen al mismo experimento. De las consultas,
el codificador recibirá solamente la columna `consulta`.

Las huellas SHA-256 identifican los archivos exactos que ya revisamos. Esta
celda no ejecuta ningún codificador.

In [1]:
import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import faiss

VERSION = "RECUPERACION_DENSA_V1"
CLAVE = "bge_m3"  # Primera incorporación; los demás modelos requieren su adaptador.
REPETICIONES = 10
SHA_CATALOGO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"
SHA_CONSULTAS = "1e35b789207b49ebb3e1e088c342cda234c2153f7d1b50f0d05adbf8cf459f17"

def sha_archivo(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()

def sha_json(objeto):
    contenido = json.dumps(objeto, ensure_ascii=False, sort_keys=True,
                           separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(contenido.encode("utf-8")).hexdigest()

inicio_lectura = perf_counter()
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/consultas_evaluacion_v1.csv").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta dentro de Tesis_Multimodelo.")

RAG = REPO / "07_RAG"
PRUEBAS = RAG / "Pruebas_RAG"
RUTA_CATALOGO = REPO / "06_grafos/catalogo_articulos.csv"
RUTA_CONSULTAS = RAG / "consultas_evaluacion_v1.csv"
RUTA_BM25 = PRUEBAS / "bm25_resultados_v1.json"
RUTA_LIBRETA = PRUEBAS / "14_recuperacion_densa.ipynb"
RUTA_SALIDA = PRUEBAS / "densa_resultados_v1.json"
assert sha_archivo(RUTA_CATALOGO) == SHA_CATALOGO, "Cambió el catálogo."
assert sha_archivo(RUTA_CONSULTAS) == SHA_CONSULTAS, "Cambiaron las consultas."

catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, keep_default_na=False)
consultas = pd.read_csv(RUTA_CONSULTAS, dtype=str, keep_default_na=False)
assert len(catalogo) == 519 and catalogo["article_id"].is_unique
assert catalogo["abstract"].str.strip().ne("").all()
assert catalogo["abstract"].nunique() == 514
assert len(consultas) == 24 and consultas["query_id"].is_unique
assert consultas["necesidad_id"].nunique() == 12
assert consultas["sha256_catalogo"].eq(SHA_CATALOGO).all()
ids = catalogo["article_id"].tolist()
ids_numericos = np.asarray([int(x) for x in ids], dtype=np.int64)
assert len(set(ids_numericos.tolist())) == 519
campos_consulta = ["query_id", "necesidad_id", "division", "idioma", "consulta"]
consultas_registro = consultas[campos_consulta].to_dict("records")

bm25 = json.loads(RUTA_BM25.read_text(encoding="utf-8"))
assert bm25["fuentes"]["catalogo"]["sha256"] == SHA_CATALOGO
assert bm25["fuentes"]["consultas"]["sha256"] == SHA_CONSULTAS
assert [r["query_id"] for r in bm25["resultados"]] == consultas["query_id"].tolist()
assert all(len(r["documentos"]) == 519 for r in bm25["resultados"])
# Se verifica la referencia, pero sus puntuaciones NO participan en esta búsqueda.
referencia_bm25 = {"ruta": "07_RAG/Pruebas_RAG/bm25_resultados_v1.json",
                  "sha256_archivo_al_inicio": sha_archivo(RUTA_BM25),
                  "sha256_resultados": bm25["sha256_resultados"],
                  "utilizado_para_puntuar": False}
del bm25

libreta = json.loads(RUTA_LIBRETA.read_text(encoding="utf-8"))
sha_codigo = sha_json(["".join(c["source"]) for c in libreta["cells"]
                      if c["cell_type"] == "code"])
lectura_validacion_ms = 1000 * (perf_counter() - inicio_lectura)
print("Catálogo:", len(catalogo), "| Consultas:", len(consultas),
      "| Necesidades:", consultas["necesidad_id"].nunique())
print("Referencias verificadas. BM25 se conserva como sistema independiente.")

Catálogo: 519 | Consultas: 24 | Necesidades: 12
Referencias verificadas. BM25 se conserva como sistema independiente.


## 2. Reutilizar los vectores y el índice de BGE-M3

Se comprueban modelo, revisión, precisión, dimensiones, identificadores y correspondencia exacta entre NPZ y FAISS. Leer `reconstruct_n` recupera los
vectores ya almacenados: **no ejecuta el codificador ni crea nuevos embeddings**.

In [2]:
RUTA_NPZ = RAG / "bge_m3_vectores_v1.npz"
RUTA_INDICE = RAG / "bge_m3_indice_v1.faiss"
SHA_NPZ = "5c0b99c8383111e1eb5bca9a38fab3b539d3b27cf64c87a886413ff3793b4feb"
SHA_INDICE = "b26b28a97d0b4b37f0c58a23d5c9e1d9859e80414f0ac743d47e3cd596c4718b"
MODELO = "BAAI/bge-m3"
REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
DIMENSION = 1024

inicio_indice = perf_counter()
assert sha_archivo(RUTA_NPZ) == SHA_NPZ, "Cambió el NPZ de artículos."
assert sha_archivo(RUTA_INDICE) == SHA_INDICE, "Cambió el índice de artículos."
with np.load(RUTA_NPZ, allow_pickle=False) as archivo:
    meta = {k: archivo[k].item() for k in archivo.files if archivo[k].ndim == 0}
    assert archivo["article_ids"].tolist() == ids, "Cambió el orden de los IDs."
    matriz_articulos = archivo["vectores"].copy()
assert meta["sha256_catalogo"] == SHA_CATALOGO
assert meta["modelo"] == MODELO and meta["modelo_revision"] == REVISION
assert meta["campo"] == "abstract" and meta["dtype_modelo"] == "torch.float32"
assert meta["prefijo_documento"] == "" and meta["prefijo_consulta"] == ""
assert int(meta["max_seq_length"]) == 8192
assert matriz_articulos.shape == (519, DIMENSION)
assert matriz_articulos.dtype == np.float32 and np.isfinite(matriz_articulos).all()
assert np.allclose(np.linalg.norm(matriz_articulos, axis=1), 1, rtol=0, atol=1e-5)

indice = faiss.read_index(str(RUTA_INDICE))
assert isinstance(indice, faiss.IndexFlatIP)
assert indice.ntotal == 519 and indice.d == DIMENSION
assert indice.metric_type == faiss.METRIC_INNER_PRODUCT
assert np.array_equal(indice.reconstruct_n(0, 519), matriz_articulos)
del matriz_articulos
faiss.omp_set_num_threads(1)  # Condición explícita para una colección pequeña.
lectura_indice_ms = 1000 * (perf_counter() - inicio_indice)

condicion = {
    "protocolo": VERSION, "modelo": MODELO, "revision": REVISION,
    "dtype_pesos": "float32", "dimension": DIMENSION, "campo_documento": "abstract",
    "campo_consulta": "consulta", "prefijo_consulta": "", "pooling": "cls",
    "max_seq_length": 8192, "batch_size_consultas": 1, "dispositivo_codificador": "cuda",
    "normalizacion": "encode normalize_embeddings=True; ajuste L2 final float32",
    "indice": "IndexFlatIP CPU", "hilos_faiss": 1, "profundidad": 519,
    "empates": "coseno completo descendente; article_id entero ascendente",
    "umbral_coseno": None, "calentamientos_por_consulta": 1,
    "repeticiones_por_consulta": REPETICIONES,
    "ranking_guardado": "primera medicion despues del calentamiento",
    "qrels_utilizados": False, "metricas_calidad_calculadas": False,
}
fuentes_comunes = {"sha256_catalogo": SHA_CATALOGO, "sha256_consultas": SHA_CONSULTAS}
fuentes_modelo = {"npz": RUTA_NPZ.name, "sha256_npz": SHA_NPZ,
                  "indice": RUTA_INDICE.name, "sha256_indice": SHA_INDICE}
firma_condicion = sha_json({"condicion": condicion, "datos": fuentes_comunes,
                           "archivos_modelo": fuentes_modelo})
sha_salida_al_inicio = sha_archivo(RUTA_SALIDA) if RUTA_SALIDA.exists() else None
if RUTA_SALIDA.exists():
    registro = json.loads(RUTA_SALIDA.read_text(encoding="utf-8"))
    assert registro["version_esquema"] == VERSION
    assert registro["fuentes"] == fuentes_comunes
    assert registro["consultas"] == consultas_registro
else:
    registro = {
        "version_esquema": VERSION, "fuentes": fuentes_comunes,
        "consultas": consultas_registro, "sistemas": {},
        "nota": "Resultados de recuperacion; pertinencia humana aun no anotada.",
    }

reutilizar = CLAVE in registro["sistemas"]
if reutilizar:
    resultado_modelo = registro["sistemas"][CLAVE]
    assert resultado_modelo["firma_condicion"] == firma_condicion, "La condición guardada difiere."
    assert sha_json(resultado_modelo["resultados"]) == resultado_modelo["sha256_resultados"]
    print("BGE-M3 ya está registrado: se reutilizará sin nuevas inferencias ni tiempos.")
else:
    print("Índice verificado:", indice.ntotal, "artículos |", indice.d, "dimensiones")
    print("Se codificarán las 24 consultas; los artículos ya están representados.")

Índice verificado: 519 artículos | 1024 dimensiones
Se codificarán las 24 consultas; los artículos ya están representados.


## 3. Preparar la búsqueda y el orden de resultados

FAISS devuelve posiciones de filas. Estas se convierten a los `article_id` guardados en el NPZ.

In [3]:
def normalizar_consulta(salida):
    # El ajuste final coincide con el aplicado a los documentos ya guardados.
    vector = np.array(salida, dtype=np.float32, order="C", copy=True)
    if vector.shape != (1, DIMENSION) or not np.isfinite(vector).all():
        raise ValueError("Vector de consulta con forma o valores incorrectos.")
    norma = np.linalg.norm(vector, axis=1)
    if not np.isfinite(norma).all() or np.any(norma <= 0):
        raise ValueError("Vector de consulta nulo o no finito.")
    faiss.normalize_L2(vector)
    if not np.allclose(np.linalg.norm(vector, axis=1), 1, rtol=0, atol=1e-5):
        raise ValueError("La consulta no quedó normalizada.")
    return vector

def buscar_vector(vector):
    scores, posiciones = indice.search(vector, indice.ntotal)
    scores, posiciones = scores[0], posiciones[0]
    # Orden determinista usando los IDs reales, no el orden de los empates de FAISS.
    orden = np.lexsort((ids_numericos[posiciones], -scores))
    return scores[orden], posiciones[orden]

def resumen_tiempos(muestras):
    return {"muestras_ms": muestras, "mediana_ms": float(np.median(muestras)),
            "minimo_ms": min(muestras), "maximo_ms": max(muestras)}

## 4. Cargar únicamente el codificador de consultas

In [4]:
if not reutilizar:
    import torch
    import transformers
    import sentence_transformers
    import huggingface_hub
    from sentence_transformers import SentenceTransformer

    if not torch.cuda.is_available():
        raise RuntimeError("Este experimento requiere el kernel entorno_tesis con CUDA.")
    versiones = {"torch": torch.__version__, "transformers": transformers.__version__,
                 "sentence_transformers": sentence_transformers.__version__}
    for nombre, actual in versiones.items():
        esperado = meta[nombre + "_version"]
        if actual != esperado:
            raise RuntimeError(f"{nombre}: se usó {esperado} y ahora hay {actual}. "
                               "Conserva el entorno original antes de ejecutar.")

    def cargar_bge():
        modelo = SentenceTransformer(
            MODELO, revision=REVISION, device="cuda", trust_remote_code=False,
            model_kwargs={"torch_dtype": torch.float32},
        )
        modelo.eval()
        assert all(p.dtype == torch.float32 and p.device.type == "cuda"
                   for p in modelo.parameters())
        assert modelo.get_sentence_embedding_dimension() == DIMENSION
        assert int(modelo.max_seq_length) == 8192
        assert getattr(modelo, "truncate_dim", None) is None
        assert getattr(modelo, "default_prompt_name", None) is None
        assert [type(mod).__name__ for mod in modelo] == ["Transformer", "Pooling", "Normalize"]
        configuracion_pooling = modelo[1].get_config_dict()
        modo = configuracion_pooling.get("pooling_mode")
        if modo is None:  # Compatibilidad con la configuración antigua del modelo.
            equivalencias = {
                "pooling_mode_cls_token": "cls", "pooling_mode_mean_tokens": "mean",
                "pooling_mode_max_tokens": "max", "pooling_mode_lasttoken": "lasttoken",
                "pooling_mode_mean_sqrt_len_tokens": "mean_sqrt_len_tokens",
                "pooling_mode_weightedmean_tokens": "weightedmean",
            }
            modos = [v for k, v in equivalencias.items() if configuracion_pooling.get(k, False)]
        else:
            modos = [modo] if isinstance(modo, str) else list(modo)
        assert modos == ["cls"], "El pooling cargado no coincide con BGE-M3."
        backbone = getattr(modelo, "transformers_model", None)
        if backbone is None:
            backbone = getattr(modelo[0], "auto_model", None)
        config = getattr(backbone, "config", None)
        revision_reportada = getattr(config, "_commit_hash", None)
        if revision_reportada is not None:
            assert revision_reportada == REVISION
        info = {"revision_solicitada": REVISION, "revision_reportada": revision_reportada,
                "dtype_modelo": str(next(modelo.parameters()).dtype),
                "pooling": configuracion_pooling,
                "atencion_reportada": getattr(config, "_attn_implementation", None),
                "prompt_encode": "", "max_seq_length": int(modelo.max_seq_length)}
        return modelo, info

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    gpu_libre_antes, gpu_total = torch.cuda.mem_get_info()
    inicio_carga = perf_counter()
    codificador, info_modelo = cargar_bge()
    torch.cuda.synchronize()
    carga_modelo_ms = 1000 * (perf_counter() - inicio_carga)

    # Solo se tokenizan las 24 consultas para comprobar que ninguna será truncada.
    tokens = codificador.tokenizer(consultas["consulta"].tolist(),
                                  add_special_tokens=True, padding=False, truncation=False)
    longitudes_consultas = [len(x) for x in tokens["input_ids"]]
    del tokens
    if max(longitudes_consultas) > 8192:
        raise ValueError("Una consulta excede el límite. No se ha truncado ni codificado.")

    entorno = {
        "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
        "faiss": getattr(faiss, "__version__", "no_declarada"), **versiones,
        "huggingface_hub": huggingface_hub.__version__,
        "sistema": platform.platform(), "procesador": platform.processor(),
        "gpu": torch.cuda.get_device_name(0), "cuda_runtime": torch.version.cuda,
        "hilos_faiss": faiss.omp_get_max_threads(), "hilos_torch_cpu": torch.get_num_threads(),
        "matmul_precision": torch.get_float32_matmul_precision(),
        "tf32_matmul_permitido": torch.backends.cuda.matmul.allow_tf32,
        "tf32_cudnn_permitido": torch.backends.cudnn.allow_tf32,
    }
    print("Modelo cargado:", MODELO, "| GPU:", entorno["gpu"])
    print("Revisión solicitada:", REVISION)
    print("Consulta sin prefijo | Tokens mín./máx.:",
          min(longitudes_consultas), max(longitudes_consultas))
else:
    print("Se conserva la ejecución guardada; no se carga el codificador.")

W1108 15:56:36.501000 13224 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Modelo cargado: BAAI/bge-m3 | GPU: NVIDIA GeForce RTX 4070 SUPER
Revisión solicitada: 5617a9f61b028005a4858fdac845db406aefb181
Consulta sin prefijo | Tokens mín./máx.: 17 37


C:\Users\hazar\AppData\Local\Temp\ipykernel_13224\2141699377.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert modelo.get_sentence_embedding_dimension() == DIMENSION


## 5. Ejecutar las consultas y medir tiempos

Se procesa una consulta a la vez (`batch_size=1`). Cada consulta tiene un calentamiento y diez mediciones completas. Por ello, su codificación se repite para medir latencia, aunque se conserve un solo vector por consulta.

El resultado guardado corresponde a la primera medición después del calentamiento. Las repeticiones permiten observar variación de tiempos y posibles diferencias numéricas; no crean nuevas consultas independientes.

In [5]:
if not reutilizar:
    def recuperar_consulta(texto):
        torch.cuda.synchronize()
        inicio = perf_counter()
        with torch.inference_mode():
            salida = codificador.encode(
                [texto], batch_size=1, prompt="", normalize_embeddings=True,
                convert_to_numpy=True, show_progress_bar=False,
            )
        vector = normalizar_consulta(salida)
        torch.cuda.synchronize()
        fin_codificacion = perf_counter()
        scores, posiciones = buscar_vector(vector)
        fin_busqueda = perf_counter()
        tiempos = {
            "codificacion": 1000 * (fin_codificacion - inicio),
            "busqueda_orden": 1000 * (fin_busqueda - fin_codificacion),
            "total": 1000 * (fin_busqueda - inicio),
        }
        return vector, scores, posiciones, tiempos

    resultados = []
    mediciones = []
    for numero, consulta in enumerate(consultas_registro):
        recuperar_consulta(consulta["consulta"])  # Calentamiento fuera de las mediciones.
        muestras = {"codificacion": [], "busqueda_orden": [], "total": []}
        mayor_diferencia = 0.0
        orden_estable = True
        for repeticion in range(REPETICIONES):
            vector, scores, posiciones, tiempos = recuperar_consulta(consulta["consulta"])
            for nombre in muestras:
                muestras[nombre].append(tiempos[nombre])
            if repeticion == 0:
                vector_guardado = vector.copy()
                scores_guardados = scores.copy()
                posiciones_guardadas = posiciones.copy()
                por_fila_guardados = np.empty(519, dtype=np.float32)
                por_fila_guardados[posiciones] = scores
            else:
                por_fila = np.empty(519, dtype=np.float32)
                por_fila[posiciones] = scores
                mayor_diferencia = max(mayor_diferencia,
                                      float(np.max(np.abs(por_fila - por_fila_guardados))))
                orden_estable = orden_estable and np.array_equal(posiciones, posiciones_guardadas)

        assert sorted(posiciones_guardadas.tolist()) == list(range(519))
        assert np.isfinite(scores_guardados).all()
        assert np.all(np.abs(scores_guardados) <= 1.00001)
        resultados.append({
            "query_id": consulta["query_id"],
            "tokens_consulta": longitudes_consultas[numero],
            "vector_consulta": vector_guardado[0].tolist(),
            "documentos": [
                {"article_id": ids[int(posicion)], "score_coseno": float(score), "rank": rango}
                for rango, (score, posicion) in enumerate(
                    zip(scores_guardados, posiciones_guardadas), start=1)
            ],
        })
        mediciones.append({
            "query_id": consulta["query_id"],
            **{nombre: resumen_tiempos(valores) for nombre, valores in muestras.items()},
            "max_diferencia_coseno_repeticiones": mayor_diferencia,
            "mismo_orden_en_todas_repeticiones": bool(orden_estable),
        })
        print(f"{numero + 1:02d}/24 | {consulta['query_id']} | "
              f"mediana total: {np.median(muestras['total']):.3f} ms")

    ejecucion = {
        "fin_utc": datetime.now(timezone.utc).isoformat(), "entorno": entorno,
        "sha256_codigo": sha_codigo, "lectura_validacion_ms": lectura_validacion_ms,
        "lectura_verificacion_indice_ms": lectura_indice_ms, "carga_modelo_ms": carga_modelo_ms,
        "calentamientos_por_consulta": 1, "repeticiones_por_consulta": REPETICIONES,
        "latencias_consulta": mediciones,
        "alcance_latencia": "consulta individual: encode + L2 float32 + FAISS519 + orden por ID",
        "gpu_total_bytes": int(gpu_total), "gpu_libre_antes_carga_bytes": int(gpu_libre_antes),
        "pico_pytorch_asignado_bytes": int(torch.cuda.max_memory_allocated()),
        "pico_pytorch_reservado_bytes": int(torch.cuda.max_memory_reserved()),
        "alcance_picos": "asignador PyTorch de este proceso desde antes de cargar el modelo",
        "ram_pico_bytes": None,
    }
    resultado_modelo = {
        "firma_condicion": firma_condicion, "condicion": condicion,
        "fuentes": fuentes_modelo, "modelo_runtime": info_modelo,
        "referencia_bm25": referencia_bm25, "resultados": resultados,
        "sha256_resultados": sha_json(resultados), "ejecuciones": [ejecucion],
    }
    # Liberar los pesos después de terminar; los resultados son listas y arrays en CPU.
    del codificador
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Resultados y tiempos originales reutilizados.")

01/24 | N01-ES | mediana total: 20.205 ms
02/24 | N01-EN | mediana total: 20.170 ms
03/24 | N02-ES | mediana total: 20.926 ms
04/24 | N02-EN | mediana total: 21.019 ms
05/24 | N03-ES | mediana total: 20.307 ms
06/24 | N03-EN | mediana total: 20.726 ms
07/24 | N04-ES | mediana total: 20.207 ms
08/24 | N04-EN | mediana total: 19.882 ms
09/24 | N05-ES | mediana total: 20.564 ms
10/24 | N05-EN | mediana total: 20.514 ms
11/24 | N06-ES | mediana total: 20.091 ms
12/24 | N06-EN | mediana total: 20.542 ms
13/24 | N07-ES | mediana total: 20.615 ms
14/24 | N07-EN | mediana total: 19.977 ms
15/24 | N08-ES | mediana total: 21.293 ms
16/24 | N08-EN | mediana total: 20.996 ms
17/24 | N09-ES | mediana total: 21.084 ms
18/24 | N09-EN | mediana total: 21.559 ms
19/24 | N10-ES | mediana total: 20.783 ms
20/24 | N10-EN | mediana total: 20.253 ms
21/24 | N11-ES | mediana total: 20.335 ms
22/24 | N11-EN | mediana total: 20.766 ms
23/24 | N12-ES | mediana total: 20.795 ms
24/24 | N12-EN | mediana total: 20

## 6. Guardar un único archivo de resultados

El JSON contiene los vectores de las 24 consultas, los 12 456 pares consulta–artículo, las posiciones, la configuración y los tiempos. Los vectores de los artículos permanecen en sus archivos originales.

In [6]:
# Comprobar al final que las cuatro entradas originales siguen intactas.
for ruta, esperado in [(RUTA_CATALOGO, SHA_CATALOGO), (RUTA_CONSULTAS, SHA_CONSULTAS),
                       (RUTA_NPZ, SHA_NPZ), (RUTA_INDICE, SHA_INDICE)]:
    assert sha_archivo(ruta) == esperado, f"Una entrada cambió: {ruta.name}"

if not reutilizar:
    # Evitar perder otra ejecución escrita mientras esta libreta estaba trabajando.
    sha_actual = sha_archivo(RUTA_SALIDA) if RUTA_SALIDA.exists() else None
    if sha_actual != sha_salida_al_inicio:
        raise RuntimeError("El JSON cambió durante la ejecución. Revisa antes de guardar.")
    registro["sistemas"][CLAVE] = resultado_modelo
    temporal = RUTA_SALIDA.with_suffix(".json.tmp")
    temporal.write_text(json.dumps(registro, ensure_ascii=False, indent=2,
                                  allow_nan=False) + "\n", encoding="utf-8")
    temporal.replace(RUTA_SALIDA)

print("Resultado:", RUTA_SALIDA)
print("Sistemas registrados:", ", ".join(registro["sistemas"]))
print("Consultas BGE-M3:", len(resultado_modelo["resultados"]))
print("Puntuaciones BGE-M3:",
      sum(len(r["documentos"]) for r in resultado_modelo["resultados"]))
print("SHA-256 del JSON:", sha_archivo(RUTA_SALIDA))

Resultado: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\densa_resultados_v1.json
Sistemas registrados: bge_m3
Consultas BGE-M3: 24
Puntuaciones BGE-M3: 12456
SHA-256 del JSON: 3d78b11c9aee893b67297eec3409f2f559e7eb620fdddfb1549572cf5ade42d8


## 7. Inspeccionar una muestra de desarrollo

Las tablas muestran únicamente N01-ES y N01-EN, pertenecientes a desarrollo.
Los títulos se añaden después de recuperar, para identificar los artículos.
Esta vista no adjudica pertinencia ni se utiliza para ajustar la condición.

Una diferencia de cosenos entre modelos no indica por sí sola cuál recupera
mejor. La comparación de calidad requiere las mismas anotaciones de pertinencia.

In [7]:
titulos = catalogo.set_index("article_id")["titulo"].to_dict()
por_query = {r["query_id"]: r for r in resultado_modelo["resultados"]}
for query_id in ["N01-ES", "N01-EN"]:
    print("\n", query_id)
    print(consultas.loc[consultas["query_id"].eq(query_id), "consulta"].iloc[0])
    tabla = pd.DataFrame(por_query[query_id]["documentos"][:10])
    tabla["titulo"] = tabla["article_id"].map(titulos)
    display(tabla[["rank", "article_id", "score_coseno", "titulo"]])

print("\nEtapa completada: recuperación densa de BGE-M3.")
print("Comparte densa_resultados_v1.json para revisar esta primera ejecución.")


 N01-ES
¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?


,rank,article_id,score_coseno,titulo
0,1,117,0.628183,A retrieval-augmented-generation pipeline to h...
1,2,285,0.533679,A Systematic Literature Review of 10 years of ...
2,3,325,0.529997,The Bible as a Corpus for Language Technologies
3,4,146,0.507228,Comparative Analysis of Information Extraction...
4,5,289,0.505368,BIOMX-DB: A web application for the BIOFACQUIM...
5,6,224,0.496606,On the Application of Natural Language Process...
6,7,253,0.496344,Spatial Analysis of Student Feedback on Learni...
7,8,31,0.496086,CUDA code to generate computational models and...
8,9,336,0.495563,Web-based wide-area monitoring platform for ri...
9,10,133,0.494997,Automatic extraction of transcriptional regula...



 N01-EN
Which studies support natural-language questions about system documentation and ground the answers in retrieved reference material?


,rank,article_id,score_coseno,titulo
0,1,117,0.592627,A retrieval-augmented-generation pipeline to h...
1,2,285,0.560387,A Systematic Literature Review of 10 years of ...
2,3,146,0.530835,Comparative Analysis of Information Extraction...
3,4,454,0.522415,Testing Three Different Speech Synthesizers to...
4,5,253,0.520436,Spatial Analysis of Student Feedback on Learni...
5,6,198,0.503931,Into The Limits of Logic: Alignment Methods fo...
6,7,175,0.502966,Exploring the Behavior and Performance of Larg...
7,8,159,0.495995,Educational Chatbots: A Systematic Mapping Study
8,9,325,0.488968,The Bible as a Corpus for Language Technologies
9,10,224,0.483135,On the Application of Natural Language Process...



Etapa completada: recuperación densa de BGE-M3.
Comparte densa_resultados_v1.json para revisar esta primera ejecución.
